https://quantum.cloud.ibm.com/learning/ja/courses/basics-of-quantum-information/entanglement-in-action/qiskit-implementation

In [ ]:
from qiskit import __version__
print(__version__)

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram, array_to_latex
from qiskit.circuit.library import UGate
from math import pi
import random


In [ ]:
qubit = QuantumRegister(1, "Q")
ebit0 = QuantumRegister(1, "A")
ebit1 = QuantumRegister(1, "B")
a = ClassicalRegister(1, "a")
b = ClassicalRegister(1, "b")

# Prepare ebit used for teleportation. 
protocol = QuantumCircuit(qubit, ebit0, ebit1, a, b)
protocol.h(ebit0)
protocol.cx(ebit0, ebit1) # 制御ビット、目的ビット
protocol.barrier()
# Alice's Operation.
protocol.cx(qubit, ebit0)
protocol.h(qubit)
protocol.barrier()
# Alice mesures and sends classical bits to Bob.
protocol.measure(ebit0, a)
protocol.measure(qubit, b)
protocol.barrier()
# Bob uses the classical bits conditional apply gates.
with protocol.if_test((a, 1)): # a==1
    protocol.x(ebit1)
with protocol.if_test((b, 1)): # b==1
    protocol.z(ebit1)

display(protocol.draw(output="mpl"))

In [ ]:
random_gate = UGate(theta=random.random() * 2 * pi, phi=random.random() * 2 * pi, lam = random.random() * 2 * pi)
display(array_to_latex(random_gate.to_matrix()))


In [ ]:
test = QuantumCircuit(qubit, ebit0, ebit1, a, b)
test.append(random_gate, qubit)
test.barrier()

test = test.compose(protocol)
test.barrier()

test.append(random_gate.inverse(), ebit1)
result = ClassicalRegister(1, "Result")
test.add_register(result)
test.measure(ebit1, result)
display(test.draw(output="mpl"))

In [ ]:
result = AerSimulator().run(test).result()
statistics = result.get_counts()
display(plot_histogram(statistics))